# Task 1 — Movie Genre Classification
**CodSoft Machine Learning Internship**

Goal: predict a movie's genre from its plot summary using TF-IDF features and
classic ML classifiers (Naive Bayes, Logistic Regression, Linear SVM).

**Before running:** download `train_data.txt` from the dataset link in the
CodSoft task PDF and place it in this same folder.

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

pd.set_option("display.max_colwidth", 80)

## 1. Load the data
The dataset ships as a `:::`-delimited text file with columns
`ID`, `TITLE`, `GENRE`, `DESCRIPTION` (no header row).

In [ ]:
train_data = pd.read_csv(
    "train_data.txt",
    sep=":::",
    engine="python",
    names=["ID", "TITLE", "GENRE", "DESCRIPTION"],
)

print("Shape:", train_data.shape)
train_data.head()

## 2. Quick EDA
How many genres are there, and how balanced are they?

In [ ]:
print("Number of unique genres:", train_data["GENRE"].nunique())
train_data["GENRE"].value_counts()

In [ ]:
plt.figure(figsize=(10, 5))
train_data["GENRE"].value_counts().plot(kind="bar")
plt.title("Movies per genre")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig("genre_distribution.png", dpi=100)
plt.show()

The genres are quite imbalanced (drama/documentary dominate). Keep this in
mind when reading the per-class scores below — rare genres will naturally
score lower.

## 3. Clean the text
Lowercase, strip punctuation/numbers, collapse whitespace. Nothing fancy —
TF-IDF does most of the heavy lifting.

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


train_data["DESCRIPTION_CLEAN"] = train_data["DESCRIPTION"].apply(clean_text)
train_data[["DESCRIPTION", "DESCRIPTION_CLEAN"]].head(3)

## 4. Train / validation split
We hold out 20% of the labeled training file to validate on, since the
official `test_data.txt` doesn't ship with genre labels.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    train_data["DESCRIPTION_CLEAN"],
    train_data["GENRE"],
    test_size=0.2,
    random_state=42,
    stratify=train_data["GENRE"],
)
print("Train size:", X_train.shape[0], "| Validation size:", X_val.shape[0])

## 5. TF-IDF vectorization

In [ ]:
tfidf = TfidfVectorizer(max_features=15000, stop_words="english", ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)
print("TF-IDF matrix shape:", X_train_tfidf.shape)

## 6. Train models
We try two classifiers the task suggests and compare them: Multinomial Naive
Bayes (fast, classic baseline for text) and Logistic Regression (usually
stronger on TF-IDF features).

In [ ]:
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)
nb_preds = nb_model.predict(X_val_tfidf)

print("Naive Bayes accuracy:", accuracy_score(y_val, nb_preds))

In [ ]:
lr_model = LogisticRegression(max_iter=1000, class_weight="balanced")
lr_model.fit(X_train_tfidf, y_train)
lr_preds = lr_model.predict(X_val_tfidf)

print("Logistic Regression accuracy:", accuracy_score(y_val, lr_preds))

## 7. Evaluate the better model in detail

In [ ]:
best_preds = lr_preds if accuracy_score(y_val, lr_preds) >= accuracy_score(y_val, nb_preds) else nb_preds
best_name = "Logistic Regression" if best_preds is lr_preds else "Naive Bayes"
print(f"Best model: {best_name}\n")
print(classification_report(y_val, best_preds, zero_division=0))

## 8. Try it on a custom plot summary
Good cell to show live in your demo video.

In [ ]:
sample_plot = [
    "A young wizard discovers he has magical powers and attends a school "
    "of witchcraft, where he must confront a dark lord threatening the world."
]
sample_clean = [clean_text(t) for t in sample_plot]
sample_tfidf = tfidf.transform(sample_clean)
predicted_genre = lr_model.predict(sample_tfidf)
print("Predicted genre:", predicted_genre[0])

## 9. Save the model + vectorizer

In [ ]:
joblib.dump(lr_model, "movie_genre_model.joblib")
joblib.dump(tfidf, "tfidf_vectorizer.joblib")
print("Saved movie_genre_model.joblib and tfidf_vectorizer.joblib")

## Conclusion
A TF-IDF + Logistic Regression pipeline gives a solid baseline for genre
classification from plot text alone. Next steps you could mention in your
video/README: try word embeddings (Word2Vec/GloVe), balance rare genres with
oversampling, or fine-tune a small transformer for a stronger score.